# Shard collection: Distributed probe data collection

Generates reasoning traces and extracts hidden states for a **subset** of AIME problems, producing one `shard_<NAME>.h5` file.

Each person on a disjoint problem range, in parallel. The resulting shards are later merged by `shard_merge.ipynb` and used to train the probe in `probe_train.ipynb`. The CoT traces saved inside each shard are also reused by the downstream segmenter task.

## Workflow at a glance

```
[Person 1] shard_collect.ipynb (problems   0-249) → shard_1.h5  ─┐
[Person 2] shard_collect.ipynb (problems 250-499) → shard_2.h5  ─┤
[Person 3] shard_collect.ipynb (problems 500-749) → shard_3.h5  ─┼ → shard_merge.ipynb → X.pt, y.pt, problem_ids.pt, traces_all.h5
[Person 4] shard_collect.ipynb (problems 750-999) → shard_4.h5  ─┘                      ↓
                                                                                     probe_train.ipynb → linear_probe.pt
```

## Suggested problem splits

*No two people should have overlap*

| # People | Person | START_IDX | END_IDX |
|---|---|---|---|
| 4 | Avni Badiwale | 0 | 250 | 
|   | B | 250 | 500 |
|   | C | 500 | 750 |
|   | D | 750 | 1000 |

---
## What this saves

`shard_<NAME>.h5` — HDF5 file, one group per problem keyed `problem_{id}`:

- `hidden_states` — `float32 [n_chunks, 3584]`. Hidden states at paragraph breaks in the CoT (layer 28, last token, subsampled to ≤ `MAX_HIDDEN_PER_PROBLEM`).
- `cot_trace` — string. Decoded tokens between `<think>...</think>`.
- `model_answer` — string. Math-verify-parsed final answer (`<none>` if none).
- `raw_output` — string. Full generation including CoT and answer.
- `label` — int32. 1 if `model_answer` matches the gold answer, else 0.
- `attrs['truncated']` — bool. True if generation hit `MAX_NEW_TOKENS` without EOS.

Root-level HDF5 attrs store the run config (`model_id`, `start_idx`, `end_idx`, `probe_layer`, `max_new_tokens`, `hidden_dim`) for consistency checks at merge time.

Also writes `shard_<NAME>_results.csv` (per-problem grading info) and `truncated_<NAME>.txt` (problem ids that hit the token cap).

## Before you run

1. **Check which range you're gonna work on** Choose the correct range
2. **Choose a unique `NAME`** (it'll be in the filename)
3. **Set the GPU runtime:** Runtime → Change runtime type → H100 (or A100). T4 is not recommended for the 7B model at full `MAX_NEW_TOKENS`
4. **Check your compute units** before starting. A 250-problem shard runs ~5 hr on H100 (~50 units at 8.71 units/hr); budget accordingly
5. **Do not modify anything but the config cell below** Otherwise your shard won't be compatible with the others

In [ ]:
# === Colab bootstrap ===
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/Avni2000/cs639-final-project.git"
    REPO_DIR = "/content/cs639-final-project"
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull

    %cd {REPO_DIR}/probe-baseline

    OUTPUT_DIR = "/content/drive/MyDrive/cs639-outputs"
else:
    OUTPUT_DIR = "."

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"IN_COLAB={IN_COLAB}  OUTPUT_DIR={OUTPUT_DIR}")

In [ ]:
%pip install -q "transformers==4.45.0" "torch>=2.3" "datasets" "pandas" \
               "math-verify[antlr4_13_2]" "tqdm" "h5py"

## Config

In [ ]:
# ========== YOUR SETTINGS ==========
NAME        = "CHANGE_ME"    # your first name or unique id; e.g. "avni"
START_IDX   = 0              # inclusive; see split table in the intro markdown
END_IDX     = 250            # exclusive
# ====================================

# ========== SHARED CONFIG ==========
MODEL_ID               = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
HIDDEN_DIM             = 3584     # DeepSeek-R1-Distill-Qwen-7B
PROBE_LAYER            = -1       # last hidden layer (= layer 28)
MAX_NEW_TOKENS         = 16384    # cap on CoT length
MAX_HIDDEN_PER_PROBLEM = 100      # uniform-random subsample of paragraph breaks
PARAGRAPH_SEED         = 0        # RNG seed for paragraph subsampling
CSV_PATH               = "aime_historical.csv"
# =============================================================================

assert NAME != "CHANGE_ME", "Set NAME to your identifier before running."
assert 0 <= START_IDX < END_IDX <= 1000, "Invalid problem range."
print(f"{NAME}: problems {START_IDX}..{END_IDX - 1}  ({END_IDX - START_IDX} problems)")

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU required. Runtime -> Change runtime type -> GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# === Download the AIME dataset if missing, load it ===
import pandas as pd

if not os.path.exists(CSV_PATH):
    from datasets import load_dataset
    ds = load_dataset("gneubig/aime-1983-2024")
    ds["train"].to_csv(CSV_PATH)

df = pd.read_csv(CSV_PATH)
problems = df[["Question", "Answer"]].dropna().reset_index(drop=True)
assert END_IDX <= len(problems), f"END_IDX={END_IDX} exceeds dataset size {len(problems)}"
subset = problems.iloc[START_IDX:END_IDX]
print(f"Dataset size: {len(problems)} problems | your range: {len(subset)} problems")

In [ ]:
# === Load the model (~1 min download on first run, cached in /content after) ===
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s")

In [ ]:
# === Helpers (same logic as train.ipynb) ===
import random

def build_prompt(problem_text: str) -> str:
    messages = [{
        "role": "user",
        "content": (
            "Solve the following AIME problem. "
            "Your final answer must be a non-negative integer (0-999), "
            "placed inside \\boxed{}.\n\n" + problem_text
        ),
    }]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _find_subseq(seq, sub):
    n, m = len(seq), len(sub)
    for i in range(n - m + 1):
        if seq[i:i + m] == sub:
            return i
    return -1

def _paragraph_break_indices(cot_token_ids):
    """Indices within cot_token_ids whose decode completes a \\n\\n break."""
    positions = []
    cursor = 0
    for i in range(len(cot_token_ids)):
        text = tokenizer.decode(cot_token_ids[:i + 1], skip_special_tokens=False)
        hit = text.find("\n\n", cursor)
        if hit != -1:
            positions.append(i)
            cursor = hit + 2
    return positions

def get_cot_hidden_states(prompt, rng=None):
    """Returns (hidden_states [N_pos, HIDDEN_DIM], generated_text, cot_trace, truncated)."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    prompt_len = input_ids.shape[1]

    # Hook captures only PROBE_LAYER's last-token hidden state per generation step and
    # moves it to CPU immediately, so GPU memory never accumulates the full sequence.
    # output_hidden_states=True would store all 28 layers x all tokens on GPU until generation ends.
    captured = []
    def _hook(module, input, output):
        captured.append(output[0][:, -1, :].detach().float().cpu())
    hook = model.model.layers[PROBE_LAYER].register_forward_hook(_hook)

    try:
        with torch.no_grad():
            gen = model.generate(
                input_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                return_dict_in_generate=True,
            )
    finally:
        hook.remove()

    generated_ids = gen.sequences[0, prompt_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=False)
    gen_list = generated_ids.tolist()

    truncated = (
        generated_ids.shape[0] >= MAX_NEW_TOKENS
        and int(generated_ids[-1]) != tokenizer.eos_token_id
    )

    hs = torch.cat(captured, dim=0) if captured else torch.empty((0, HIDDEN_DIM), dtype=torch.float32)

    think_toks = tokenizer.encode("<think>",  add_special_tokens=False)
    end_toks   = tokenizer.encode("</think>", add_special_tokens=False)
    tp = _find_subseq(gen_list, think_toks)
    ep = _find_subseq(gen_list, end_toks)
    if tp != -1 and ep != -1 and ep > tp:
        cot_start, cot_end = tp + len(think_toks), ep
    else:
        cot_start, cot_end = 0, len(gen_list)

    cot_ids = gen_list[cot_start:cot_end]
    cot_trace = tokenizer.decode(cot_ids, skip_special_tokens=False)

    rel = _paragraph_break_indices(cot_ids)
    if len(rel) > MAX_HIDDEN_PER_PROBLEM:
        r = rng if rng is not None else random
        rel = sorted(r.sample(rel, MAX_HIDDEN_PER_PROBLEM))

    if rel:
        abs_pos = torch.tensor([cot_start + p for p in rel], dtype=torch.long)
        return hs.index_select(0, abs_pos), generated_text, cot_trace, truncated
    return torch.empty((0, HIDDEN_DIM), dtype=torch.float32), generated_text, cot_trace, truncated

In [ ]:
# === Collection loop ===
import h5py
from math_verify import parse, verify
from tqdm.auto import tqdm

rng = random.Random(PARAGRAPH_SEED)
shard_path    = os.path.join(OUTPUT_DIR, f"shard_{NAME}.h5")
results_path  = os.path.join(OUTPUT_DIR, f"shard_{NAME}_results.csv")
truncated_log = os.path.join(OUTPUT_DIR, f"truncated_{NAME}.txt")

STR_DT = h5py.string_dtype(encoding='utf-8')

def existing_ids(path):
    if not os.path.exists(path):
        return set()
    with h5py.File(path, 'r') as f:
        return {int(k.split('_', 1)[1]) for k in f.keys() if k.startswith('problem_')}

done = existing_ids(shard_path)
todo = subset[~subset.index.isin(done)]

if done:
    next_idx = todo.index.min() if len(todo) else "<nothing left>"
    print(f"Resuming: {len(done)} done, next={next_idx}, {len(todo)} remaining in [{START_IDX}, {END_IDX})")
else:
    print(f"Starting fresh: {len(todo)} problems in [{START_IDX}, {END_IDX})")

results = []
t_start = time.time()

for pos, (idx, row) in enumerate(tqdm(todo.iterrows(), total=len(todo), desc=f"{NAME} collecting")):
    prompt = build_prompt(row["Question"])
    t0 = time.time()
    cot_hidden, gen_text, cot_trace, truncated = get_cot_hidden_states(prompt, rng=rng)
    wall = time.time() - t0

    if truncated:
        print(f"  !! problem {idx}: hit MAX_NEW_TOKENS={MAX_NEW_TOKENS} before EOS")
        with open(truncated_log, "a") as lf:
            lf.write(f"{idx}\n")

    gold = parse(f"${row['Answer']}$")
    pred = parse(gen_text)
    is_correct = bool(verify(gold, pred)) if pred else False
    label = 1 if is_correct else 0
    pred_str = str(pred[0]) if pred else "<none>"

    with h5py.File(shard_path, 'a') as f:
        key = f"problem_{idx}"
        if key in f:
            del f[key]
        grp = f.create_group(key)
        grp.create_dataset('hidden_states', data=cot_hidden.numpy())
        grp.create_dataset('cot_trace',    data=cot_trace,         dtype=STR_DT)
        grp.create_dataset('model_answer', data=pred_str,          dtype=STR_DT)
        grp.create_dataset('raw_output',   data=gen_text,          dtype=STR_DT)
        grp.create_dataset('label',        data=label,             dtype='int32')
        grp.create_dataset('truncated',    data=int(truncated),    dtype='int32')
        f.flush()

    results.append({
        "problem_idx": int(idx),
        "correct_answer": str(row["Answer"]),
        "predicted": pred_str,
        "label": label,
        "positions": cot_hidden.shape[0],
        "wall_s": wall,
        "truncated": truncated,
    })
    print(f"  [{len(done)+pos+1}/{len(subset)}] problem_idx={idx} label={label} "
          f"positions={cot_hidden.shape[0]} wall={wall:.1f}s truncated={truncated}")

print(f"\nTotal wall time: {(time.time() - t_start) / 60:.1f} min")
print(f"Correct / total (this run): {sum(r['label'] for r in results)} / {len(results)}")

In [ ]:
# === Finalize shard ===
# Write config attrs so shard_merge.ipynb can verify consistency across shards
with h5py.File(shard_path, 'a') as f:
    f.attrs['model_id']       = MODEL_ID
    f.attrs['name']           = NAME
    f.attrs['start_idx']      = START_IDX
    f.attrs['end_idx']        = END_IDX
    f.attrs['probe_layer']    = PROBE_LAYER
    f.attrs['max_new_tokens'] = MAX_NEW_TOKENS
    f.attrs['hidden_dim']     = HIDDEN_DIM

pd.DataFrame(results).to_csv(results_path, index=False)

n_problems = len(existing_ids(shard_path))
print(f"✓ Shard saved    -> {shard_path}  ({n_problems} problems)")
print(f"✓ Results saved  -> {results_path}")
if os.path.exists(truncated_log):
    with open(truncated_log) as lf:
        n_trunc = sum(1 for _ in lf)
    print(f"  {n_trunc} truncated problem(s) logged -> {truncated_log}")

## Next steps

1. **Locate your shard file:** `MyDrive/cs639-outputs/shard_<NAME>.h5`
2. **Share it with the merge-runner:** Upload your shard to the shared Drive folder.
3. **Upload your notebook in cs639-final-project/probe-baseline/shards-logs** to record your label distribution.

### Troubleshooting

- **OOM during generation:** Lower `MAX_NEW_TOKENS` in the config cell and coordinate with the team so everyone uses the same value.
- **Colab disconnected mid-run:** Re-run the collection loop cell. It will detect already-written problems in the `.h5` file and resume where it left off — no work is lost.